In [1]:
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.util import fetch_wasabi_file
from histpy import Histogram

from scoords import SpacecraftFrame

from astropy.time import Time
import astropy.units as u

SED_KEV_TO_ERG = u.keV.to(u.erg)
KEV_TO_MEV = u.keV.to(u.MeV)
from astropy.coordinates import SkyCoord, Galactic

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from threeML import *
from threeML.io.package_data import get_path_of_data_file
from threeML.io.logging import silence_console_log
from astromodels import Parameter
from threeML.minimizer.minimization import CannotComputeCovariance

from jupyterthemes import jtplot
jtplot.style(context="talk", fscale=1, ticks=True, grid=False)
set_threeML_style()
silence_warnings()

from scipy.integrate import quad

import matplotlib.ticker as mticker

from pathlib import Path

import os

%matplotlib inline

12:02:54 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=277251;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=513898;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=74578;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=523904;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

12:02:55 WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=161719;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=885526;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

12:02:55 INFO      Starting 3ML!                                                                     ]8;id=158313;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=586325;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=610733;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=908916;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=484072;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=460711;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

12:02:56 WARNING   No fermitools installed                                              ]8;id=249985;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=220323;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable OMP_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=177539;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=829259;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=418417;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=495649;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=87028;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=170111;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

In [2]:
data_path = Path("/Users/parshadkp/Software/COSI_Data/")

In [3]:
from cosipy.event_selection import GoodTimeInterval
from agn_cosi_fit_utils import open_spacecraft_history, scale_spacecraft_livetime

orientation_path = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
source_coord = SkyCoord(l=172.104, b=-51.934, frame="galactic", unit="deg")
fov_cut = 60 * u.deg

full_sc_orientation = open_spacecraft_history(orientation_path)
source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    full_sc_orientation,
    fov_cut,
    earth_occ=False,
)
sc_orientation = full_sc_orientation.apply_gti(source_gti)

print(f"NGC 1068 FOV cut: {fov_cut.to_value(u.deg):.0f} deg")
print(f"Selected livetime: {sc_orientation.cumulative_livetime().to_value(u.s):,.1f} s")


NGC 1068 FOV cut: 60 deg
Selected livetime: 2,546,115.0 s


In [4]:
dr = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"

In [5]:
multiplier_1068 = 8
exposure_1068 = multiplier_1068 * 3

# Scale count histograms and response livetime together; keep source flux intrinsic.
sc_orientation = scale_spacecraft_livetime(sc_orientation, multiplier_1068)


# NGC 1068 CPL + PL comparison


## Injected cutoff power law (thermal)


### NGC 1068 ($E_c=128$ keV)


In [6]:
K_inj = 3.1e-1 / u.cm / u.cm / u.s / u.keV
piv_inj = 1.0 * u.keV
xc_inj = 128.0 * u.keV
index_inj = -2.10

spectrum_inj_1068 = Cutoff_powerlaw()
spectrum_inj_1068.K.value = K_inj.value
spectrum_inj_1068.piv.value = piv_inj.value
spectrum_inj_1068.xc.value = xc_inj.value
spectrum_inj_1068.index.value = index_inj
spectrum_inj_1068.K.unit = K_inj.unit
spectrum_inj_1068.piv.unit = piv_inj.unit
spectrum_inj_1068.xc.unit = xc_inj.unit


def cutoff_powerlaw_k_at_pivot(shape, pivot_value):
    """Return the equivalent CPL K after changing its pivot."""
    return float(shape.K.value * (pivot_value / shape.piv.value) ** shape.index.value)


## Injected power-law tail (non-thermal)


In [36]:
norm_nt = 0.42
tail_pivot_keV = 200.0
tail_index = -2.8

K_inj_tail = (
    norm_nt * spectrum_inj_1068.evaluate_at(tail_pivot_keV)
    / u.cm / u.cm / u.s / u.keV
)

spectrum_inj_1068_PL = Powerlaw()
spectrum_inj_1068_PL.K.value = K_inj_tail.value
spectrum_inj_1068_PL.piv.value = tail_pivot_keV
spectrum_inj_1068_PL.index.value = tail_index
spectrum_inj_1068_PL.K.unit = K_inj_tail.unit
spectrum_inj_1068_PL.piv.unit = u.keV

spectrum_inj_1068_total = spectrum_inj_1068 + spectrum_inj_1068_PL

# The linked fit uses both component normalizations at a 200 keV pivot.
thermal_k_at_200 = cutoff_powerlaw_k_at_pivot(spectrum_inj_1068, tail_pivot_keV)
linking_ratio_1068 = spectrum_inj_1068_PL.K.value / thermal_k_at_200

print("Thermal CPL K at a 200 keV pivot:", thermal_k_at_200)
print("Injected power-law K at 200 keV:", spectrum_inj_1068_PL.K.value)
print("Linked K ratio:", linking_ratio_1068)
print(
    "Flux-density ratio at 200 keV:",
    spectrum_inj_1068_PL.evaluate_at(200) / spectrum_inj_1068.evaluate_at(200),
)

thermal_energy_flux, _ = quad(
    lambda energy: energy * spectrum_inj_1068.evaluate_at(energy),
    100.0,
    10000.0,
)
nonthermal_energy_flux, _ = quad(
    lambda energy: energy * spectrum_inj_1068_PL.evaluate_at(energy),
    100.0,
    10000.0,
)
print(
    "Non-thermal 0.1-10 MeV energy-flux fraction:",
    nonthermal_energy_flux / (thermal_energy_flux + nonthermal_energy_flux),
)


Thermal CPL K at a 200 keV pivot: 4.562456144556676e-06
Injected power-law K at 200 keV: 4.0166395973616155e-07
Linked K ratio: 0.08803678260346114
Flux-density ratio at 200 keV: 0.4200000000000001
Non-thermal 0.1-10 MeV energy-flux fraction: 0.36317938227868196


# Spectral fitting


In [37]:
source_file = (
    data_path
    / "AGN_Data/GammaRay/Paper_Models"
    / "NGC1068_ec_128_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p42.hdf5"
)
background_file = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Software_Files/DC4_Files/Background/"
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC1068_60deg_fov_cut.hdf5"
)

for required_file in (source_file, background_file):
    if not required_file.exists():
        raise FileNotFoundError(required_file)

ngc1068_source_hist = Histogram.open(source_file) * multiplier_1068
bkg = Histogram.open(background_file) * multiplier_1068

# Collapse the background time axis and match the source histogram metadata.
bkg = bkg.project("Em", "Phi", "PsiChi")
ngc1068_source_hist.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
ngc1068_source_hist = ngc1068_source_hist.to(unit=bkg.unit, update=False)
ngc1068_data_hist = ngc1068_source_hist + bkg

print(f"Loaded source: {source_file.name}")
print(f"Loaded background: {background_file.name}")


Loaded source: NGC1068_ec_128_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p42.hdf5
Loaded background: Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_NGC1068_60deg_fov_cut.hdf5


## Perform the COSI-only spectral fits


Fit the injected CPL+PL model and a nested CPL-only model for comparison.


In [38]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter

bkg_par = Parameter(
    "background_cosi",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the NGC 1068 COSI fit",
)

cosi = COSIPlugin(
    "cosi",
    dr=dr,
    data=ngc1068_data_hist.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par,
    earth_occ=True,
)


### Thermal cutoff-power-law component


In [39]:
l = 172.104
b = -51.934

# Express the injected CPL at the 200 keV pivot used by the fit.
K = cutoff_powerlaw_k_at_pivot(spectrum_inj_1068, 200.0) / u.cm / u.cm / u.s / u.keV
piv = 200.0 * u.keV
xc = 128.0 * u.keV
index = -2.10

spectrum_cpl = Cutoff_powerlaw()
spectrum_cpl.K.value = K.value
spectrum_cpl.piv.value = piv.value
spectrum_cpl.xc.value = xc.value
spectrum_cpl.index.value = index
spectrum_cpl.index.fix = True

spectrum_cpl.K.min_value = 1e-8
spectrum_cpl.K.max_value = 1e-2
spectrum_cpl.xc.min_value = 10
spectrum_cpl.xc.max_value = 2000

spectrum_cpl.K.unit = K.unit
spectrum_cpl.piv.unit = piv.unit
spectrum_cpl.xc.unit = xc.unit


### Thermal + non-thermal model


In [40]:
K_tail = spectrum_inj_1068_PL.K.value / u.cm / u.cm / u.s / u.keV
piv_tail = 200.0 * u.keV
index_tail = -2.8

spectrum_pl = Powerlaw()
spectrum_pl.K.value = K_tail.value
spectrum_pl.piv.value = piv_tail.value
spectrum_pl.index.value = index_tail

spectrum_pl.K.min_value = 1e-10
spectrum_pl.K.max_value = 1e-4
spectrum_pl.index.min_value = -5
spectrum_pl.index.max_value = 1
spectrum_pl.index.delta = 0.25

spectrum_pl.K.unit = K_tail.unit
spectrum_pl.piv.unit = piv_tail.unit


In [41]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter

# Keep both components in one point source so response caching follows linked parameters.
ngc1068 = PointSource(
    "NGC1068",
    l=l,
    b=b,
    spectral_shape=spectrum_cpl + spectrum_pl,
)
model = Model(ngc1068)
cosi.set_model(model)


def make_cpl_only_source(name, reference_source):
    reference_shape = reference_source.spectrum.main.shape.functions[0]
    cpl_shape = Cutoff_powerlaw()

    for parameter_name in ("K", "piv", "xc", "index"):
        reference_parameter = getattr(reference_shape, parameter_name)
        cpl_parameter = getattr(cpl_shape, parameter_name)
        cpl_parameter.value = reference_parameter.value
        cpl_parameter.fix = reference_parameter.fix

        if reference_parameter.min_value is not None:
            cpl_parameter.min_value = reference_parameter.min_value
        if reference_parameter.max_value is not None:
            cpl_parameter.max_value = reference_parameter.max_value
        if reference_parameter.delta is not None:
            cpl_parameter.delta = reference_parameter.delta
        if reference_parameter.unit is not None:
            cpl_parameter.unit = reference_parameter.unit

    return PointSource(name, l=l, b=b, spectral_shape=cpl_shape)


ngc1068_cpl_only = make_cpl_only_source("NGC1068_cpl_only", ngc1068)
model_cpl_only = Model(ngc1068_cpl_only)

bkg_par_cpl_only = Parameter(
    "background_cosi_cpl_only",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the CPL-only NGC 1068 fit",
)

cosi_cpl_only = COSIPlugin(
    "cosi_cpl_only",
    dr=dr,
    data=ngc1068_data_hist.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par_cpl_only,
    earth_occ=True,
)
cosi_cpl_only.set_model(model_cpl_only)


### COSI-only data lists


In [42]:
plugins = DataList(cosi)
plugins_cpl_only = DataList(cosi_cpl_only)


### CPL+PL versus CPL-only comparison


In [43]:
link_function = Line(a=0.0, b=linking_ratio_1068)
link_function.a.fix = True
link_function.b.min_value = 0.0

model.link(
    model["NGC1068"].spectrum.main.composite.K_2,
    model["NGC1068"].spectrum.main.composite.K_1,
    link_function,
)

like = JointLikelihood(model, plugins, verbose=False)
like_cpl_only = JointLikelihood(model_cpl_only, plugins_cpl_only, verbose=False)

result = like.fit()
result_cpl_only = like_cpl_only.fit()


12:10:51 INFO      set the minimizer to minuit                                             ]8;id=859498;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=615681;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to minuit                                             ]8;id=332022;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=110685;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
NGC1068.spectrum.main.composite.K_1,(4.6 -2.1 +4) x 10^-6,1 / (keV s cm2)
NGC1068.spectrum.main.composite.xc_1,(1.28 -0.30 +0.4) x 10^2,keV
NGC1068.spectrum.main.composite.K_2.Line.b,(0.9 +/- 1.4) x 10^-1,
NGC1068.spectrum.main.composite.index_2,-2.8 +/- 0.9,
background_cosi,(2.48983 +/- 0.00015) x 10,Hz


Correlation matrix:

...

1.00,-0.94,0.07
-0.94,1.00,-0.30
0.07,-0.30,1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi_cpl_only,-3786219925.933871
total,-3786219925.933871


Values of statistical measures:

,statistical measures
AIC,-7572439845.867638
BIC,-7572439814.825025


In [44]:
from agn_cosi_fit_utils import COSIPlugin


def make_null_likelihood(data_hist):
    bkg_par_null = Parameter(
        "background_cosi_null",
        1,
        min_value=0,
        max_value=5,
        delta=0.05,
        desc="Background parameter for the NGC 1068 null fit",
    )
    cosi_null = COSIPlugin(
        "cosi_null",
        dr=dr,
        data=data_hist.project("Em", "Phi", "PsiChi"),
        bkg=bkg.project("Em", "Phi", "PsiChi"),
        sc_orientation=sc_orientation,
        nuisance_param=bkg_par_null,
        earth_occ=True,
    )

    spectrum_null = Powerlaw()
    spectrum_null.K.value = 1e-30
    spectrum_null.index.value = 1
    spectrum_null.K.fix = True
    spectrum_null.index.fix = True

    source_null = PointSource(
        "NGC1068_null",
        l=l,
        b=b,
        spectral_shape=spectrum_null,
    )
    model_null = Model(source_null)
    cosi_null.set_model(model_null)

    like_null = JointLikelihood(model_null, DataList(cosi_null), verbose=False)
    like_null.fit()
    return like_null


def get_likelihood_statistic(joint_likelihood):
    statistic = joint_likelihood.results.get_statistic_frame()["-log(likelihood)"]
    if "total" in statistic.index:
        return float(statistic.loc["total"])
    return float(statistic.sum())


def detection_ts(null_likelihood, source_likelihood):
    return 2.0 * (
        get_likelihood_statistic(null_likelihood)
        - get_likelihood_statistic(source_likelihood)
    )


def model_improvement_ts(reference_likelihood, test_likelihood):
    return 2.0 * (
        get_likelihood_statistic(reference_likelihood)
        - get_likelihood_statistic(test_likelihood)
    )


like_null = make_null_likelihood(ngc1068_data_hist)
TS_cpl = detection_ts(like_null, like_cpl_only)
TS_cpl_pl = detection_ts(like_null, like)
TS_cpl_pl_vs_cpl = model_improvement_ts(like_cpl_only, like)

fit_ts_comparison = pd.DataFrame(
    [
        {
            "spectrum": "NGC 1068 Ec=128 keV",
            "TS_CPL": TS_cpl,
            "TS_CPL_plus_PL": TS_cpl_pl,
            "Delta_TS_CPL_plus_PL_vs_CPL": TS_cpl_pl_vs_cpl,
            "Sigma_CPL": np.sqrt(max(TS_cpl, 0.0)),
            "Sigma_CPL_plus_PL": np.sqrt(max(TS_cpl_pl, 0.0)),
            "Sigma_added_PL": np.sqrt(max(TS_cpl_pl_vs_cpl, 0.0)),
        }
    ]
)
display(fit_ts_comparison)

TS = TS_cpl_pl


12:11:10 INFO      set the minimizer to minuit                                             ]8;id=108447;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=805638;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
background_cosi_null,(2.49131 +/- 0.00011) x 10,Hz


Correlation matrix:

1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi_null,-3786219663.8776383
total,-3786219663.8776383


Values of statistical measures:

,statistical measures
AIC,-7572439325.7552595
BIC,-7572439315.407704


,spectrum,TS_CPL,TS_CPL_plus_PL,Delta_TS_CPL_plus_PL_vs_CPL,Sigma_CPL,Sigma_CPL_plus_PL,Sigma_added_PL
0,NGC 1068 Ec=128 keV,524.112465,533.2652,9.152735,22.893503,23.092536,3.025349


In [ ]:
# Save the global-fit and injected-parameter summary.
from agn_cosi_fit_utils import save_agn_fit_summary

FIT_SUMMARY_DIR = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "Papers/AGN_Corona_EC_Overleaf/Fits"
)
fit_summary_path = save_agn_fit_summary(
    output_path=(
        FIT_SUMMARY_DIR
        / f"NGC1068_Paper_Plots_TH_NTH_Comp_fit_summary_{exposure_1068}Months.txt"
    ),
    fit_results={"ngc1068_cpl_pl": like.results},
    injected_models={
        "ngc1068_cpl_pl": {
            "thermal": spectrum_inj_1068,
            "nonthermal": spectrum_inj_1068_PL,
        }
    },
    ts_values={"ngc1068_cpl_pl": TS_cpl_pl},
    exposure_months={"ngc1068_cpl_pl": exposure_1068},
    extra_statistics={
        "ngc1068_cpl_pl": {
            "TS_CPL": TS_cpl,
            "Delta_TS_CPL_plus_PL_vs_CPL": TS_cpl_pl_vs_cpl,
        }
    },
)
print(f"Saved fit summary: {fit_summary_path}")


In [ ]:
results = like.results
results_cpl_only = like_cpl_only.results
results_sed = results
data_sed_hist = cosi._data.copy()
bkg_sed_hist = cosi._bkg_hist.copy()

print(results.display())
print(results_cpl_only.display())


def make_composite_flux_propagator(fit_results, source_name):
    optimized_source = fit_results.optimized_model[source_name]
    thermal_shape, tail_shape = optimized_source.spectrum.main.shape.functions[:2]
    link_function = tail_shape.K.auxiliary_variable[1]

    def evaluate_at(energy, K_1, xc_1, b, index_2):
        thermal_flux = thermal_shape.evaluate_at(energy, K=K_1, xc=xc_1)
        tail_flux = tail_shape.evaluate_at(energy, K=b * K_1, index=index_2)
        return thermal_flux + tail_flux

    return fit_results.propagate(
        evaluate_at,
        K_1=fit_results.get_variates(thermal_shape.K.path),
        xc_1=fit_results.get_variates(thermal_shape.xc.path),
        b=fit_results.get_variates(link_function.b.path),
        index_2=fit_results.get_variates(tail_shape.index.path),
    )


results_err_ngc1068 = make_composite_flux_propagator(results, "NGC1068")


In [ ]:
energy = np.geomspace(100 * u.keV, 10 * u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)
flux_median_thermal = np.zeros_like(energy)
flux_median_tail = np.zeros_like(energy)

best_fit_shape = results.optimized_model["NGC1068"].spectrum.main.shape
best_fit_thermal, best_fit_tail = best_fit_shape.functions[:2]

for index, energy_value in enumerate(energy):
    combined_flux = results_err_ngc1068(energy_value)
    flux_median[index] = combined_flux.median
    flux_lo[index], flux_hi[index] = combined_flux.equal_tail_interval(cl=0.68)
    flux_median_thermal[index] = best_fit_thermal.evaluate_at(energy_value)
    flux_median_tail[index] = best_fit_tail.evaluate_at(energy_value)
    flux_inj[index] = spectrum_inj_1068_total.evaluate_at(energy_value)

binned_energy_edges = ngc1068_source_hist.axes["Em"].edges.value
binned_energy = 0.5 * (binned_energy_edges[:-1] + binned_energy_edges[1:])
bin_sizes = np.diff(binned_energy_edges)
expectation = cosi._expected_counts["NGC1068"]


In [ ]:
FONT_SIZE = 25
plt.rcParams["agg.path.chunksize"] = 10000
plt.rcParams.update({"font.size": FONT_SIZE})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.linewidth"] = 1.5

fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
ax.tick_params(which="major", size=12, width=1.5, direction="in", top=True, right=True)
ax.tick_params(which="minor", size=6, width=1.5, direction="in", top=True, right=True)
ax.spines["right"].set_visible(True)
ax.spines["top"].set_visible(True)

sigma = np.sqrt(max(TS, 0.0))

ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy**2 * flux_inj,
    color="#D55E00",
    ls=":",
    lw=3,
    label=r"Injected CPL + PL ($E_c=128$ keV)",
)
ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy**2 * flux_median,
    color="#D55E00",
    lw=2,
    label=fr"CPL + PL best fit ({sigma:.2f}$\sigma$, $\Delta$TS={TS_cpl_pl_vs_cpl:.2f})",
)
ax.fill_between(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy**2 * flux_lo,
    SED_KEV_TO_ERG * energy**2 * flux_hi,
    alpha=0.25,
    color="#D55E00",
    label="68% containment band",
)

ax.text(
    0.98,
    0.98,
    f"NGC 1068 (CPL + PL)\n{exposure_1068}-month",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=FONT_SIZE,
    fontweight="550",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim((100) * KEV_TO_MEV, (10000) * KEV_TO_MEV)
ax.set_ylim(5e-5 * SED_KEV_TO_ERG, 10 * SED_KEV_TO_ERG)
ax.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE)
ax.set_ylabel(r"Energy Flux (erg cm$^{-2}$ s$^{-1}$)", fontsize=FONT_SIZE)
ax.legend(fontsize=FONT_SIZE, loc="lower left", frameon=False)

save_path = (
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    f"Papers/AGN_Corona_EC/Plots/NGC1068_CPL_PL_Fit_128_{exposure_1068}Months.pdf"
)
# plt.savefig(save_path)


## NGC 1068 bin-by-bin SED

The CPL index and cutoff are frozen to the global-fit values in each bin. The linked power-law normalization follows the fitted CPL normalization.


In [ ]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
def _iter_model_sources(model):
    return [
        (name, child)
        for name, child in getattr(model, '_children', {}).items()
        if hasattr(child, 'spectrum')
    ]


def _find_shape_by_class(shape, shape_name):
    if shape is None:
        return None
    if shape.__class__.__name__ == shape_name:
        return shape

    for child_shape in getattr(shape, 'functions', []):
        match = _find_shape_by_class(child_shape, shape_name)
        if match is not None:
            return match

    return None


def _get_source_spectral_shape(source, shape_name='Cutoff_powerlaw'):
    main_component = source.spectrum.main

    for attr in (shape_name, 'shape', 'composite'):
        candidate = getattr(main_component, attr, None)
        match = _find_shape_by_class(candidate, shape_name)
        if match is not None:
            return match

    return None


def _resolve_source_name(model, source_name=None, shape_name='Cutoff_powerlaw'):
    sources = _iter_model_sources(model)

    if source_name is not None:
        source_names = [name for name, _ in sources]
        if source_name not in source_names:
            raise ValueError(f'Source {source_name!r} is not in optimized model. Available sources: {source_names}.')

        source = model[source_name]
        if _get_source_spectral_shape(source, shape_name=shape_name) is None:
            raise ValueError(f'Source {source_name!r} does not contain a {shape_name} shape.')

        return source_name

    source_names = [
        name
        for name, source in sources
        if _get_source_spectral_shape(source, shape_name=shape_name) is not None
    ]

    if len(source_names) != 1:
        raise ValueError(
            f'Expected one source with a {shape_name} shape, found {source_names}. '
            'Pass source_name to select the SED target explicitly.'
        )

    return source_names[0]


def _get_cutoff_powerlaw_shape(global_results, source_name=None):
    optimized_model = global_results.optimized_model
    source_name = _resolve_source_name(optimized_model, source_name=source_name)

    return _get_source_spectral_shape(optimized_model[source_name]), source_name


def build_thermal_sed_model(global_results, freeze_cutoff=True, source_name=None, freeze_non_target_parameters=True):
    from astromodels import clone_model

    best_fit_shape, source_name = _get_cutoff_powerlaw_shape(global_results, source_name=source_name)
    sed_model = clone_model(global_results.optimized_model)
    fit_shape = _get_source_spectral_shape(sed_model[source_name])

    if freeze_non_target_parameters:
        for parameter in sed_model.parameters.values():
            parameter.fix = True

    fit_shape.K.value = best_fit_shape.K.value
    fit_shape.K.min_value = max(best_fit_shape.K.value * 1e-3, 1e-12)
    fit_shape.K.max_value = best_fit_shape.K.value * 1e3
    fit_shape.K.fix = False
    fit_shape.index.value = best_fit_shape.index.value
    fit_shape.index.fix = True
    fit_shape.xc.value = best_fit_shape.xc.value
    fit_shape.xc.fix = freeze_cutoff

    return sed_model


def _hist_sum(hist):
    contents = hist.contents
    if hasattr(contents, 'todense'):
        contents = contents.todense()
    if hasattr(contents, 'value'):
        contents = contents.value
    return float(np.asarray(contents).sum())


def _evaluate_model_photon_flux(model, energy):
    photon_flux = 0.0

    for _, source in _iter_model_sources(model):
        photon_flux += float(source.spectrum.main.shape.evaluate_at(energy))

    return photon_flux


def _with_target_k_value(model, source_name, k_value, evaluator):
    target_shape = _get_source_spectral_shape(model[source_name])
    original_k = target_shape.K.value
    target_shape.K.value = float(k_value)

    try:
        return evaluator()
    finally:
        target_shape.K.value = original_k


def _integrate_model_energy_flux(model, source_name, k_value, e_min, e_max, n_points=256):
    energy_grid = np.geomspace(e_min, e_max, n_points)

    def evaluator():
        return np.asarray([_evaluate_model_photon_flux(model, energy) for energy in energy_grid])

    photon_flux = _with_target_k_value(model, source_name, k_value, evaluator)
    trapezoid = getattr(np, 'trapezoid', None)
    if trapezoid is None:
        trapezoid = np.trapz
    return float(trapezoid(energy_grid * photon_flux, energy_grid))


def _evaluate_model_sed_flux(model, source_name, k_value, energy):
    return _with_target_k_value(
        model,
        source_name,
        k_value,
        lambda: float(energy**2 * _evaluate_model_photon_flux(model, energy)),
    )


def _extract_profile_error(errors, suffix):
    if hasattr(errors, 'index'):
        for parameter_path in errors.index:
            if str(parameter_path).endswith(suffix):
                row = errors.loc[parameter_path]
                return float(row['negative_error']), float(row['positive_error'])
        return None

    for parameter_path, interval in errors.items():
        if str(parameter_path).endswith(suffix):
            return interval
    return None


def _get_global_background_rate(global_results):
    results_frame = global_results.get_data_frame()
    background_paths = [
        parameter_path
        for parameter_path in results_frame.index
        if 'background_' in str(parameter_path) or 'total_bkg' in str(parameter_path)
    ]

    if len(background_paths) != 1:
        raise ValueError(
            'Expected exactly one fitted background-rate parameter, '
            f'found {list(map(str, background_paths))}.'
        )

    return float(results_frame.loc[background_paths[0], 'value'])


def _run_joint_likelihood_fit(joint_likelihood):
    fit_status = 'covariance'
    try:
        joint_likelihood.fit(quiet=True)
    except Exception as exc:
        fit_status = f'no-covariance fallback ({type(exc).__name__})'
        joint_likelihood.fit(quiet=True, compute_covariance=False)
    return fit_status


def _get_joint_likelihood_statistic(joint_likelihood):
    statistic_frame = joint_likelihood.results.get_statistic_frame()

    if '-log(likelihood)' in statistic_frame.columns:
        statistic_column = statistic_frame['-log(likelihood)']
    else:
        numeric_frame = statistic_frame.select_dtypes(include=[np.number])
        if numeric_frame.empty:
            raise ValueError(f'No numeric statistic columns found in {list(statistic_frame.columns)}.')
        statistic_column = numeric_frame.iloc[:, 0]

    if 'total' in statistic_column.index:
        return float(statistic_column.loc['total'])

    return float(statistic_column.sum())


def _ts_to_sigma(ts_value):
    if not np.isfinite(ts_value) or ts_value <= 0:
        return 0.0

    return float(np.sqrt(ts_value))


def _build_null_sed_model(
    global_results,
    source_name,
    freeze_cutoff=True,
    freeze_non_target_parameters=True,
    null_k_value=1e-30,
):
    null_model = build_thermal_sed_model(
        global_results,
        freeze_cutoff=freeze_cutoff,
        source_name=source_name,
        freeze_non_target_parameters=freeze_non_target_parameters,
    )
    null_shape = _get_source_spectral_shape(null_model[source_name])
    null_k_value = max(float(null_k_value), np.finfo(float).tiny)
    null_k_min = max(null_k_value * 0.1, np.finfo(float).tiny)
    null_shape.K.min_value = min(float(null_shape.K.min_value or null_k_min), null_k_min)
    null_shape.K.value = null_k_value
    null_shape.K.fix = True

    return null_model


def _make_minuit_with_ftol(minuit_ftol):
    if minuit_ftol is None:
        return None

    minuit = LocalMinimization('minuit')
    minuit.setup(ftol=float(minuit_ftol))

    return minuit


def _hits_lower_bound(value, lower_bound):
    if lower_bound is None or not np.isfinite(lower_bound):
        return False

    atol = max(1e-18, abs(float(lower_bound)) * 1e-6)
    return bool(np.isclose(float(value), float(lower_bound), rtol=1e-3, atol=atol))


def _interpolate_profile_crossing(x1, y1, x2, y2, target):
    if np.isclose(y1, y2):
        return float(0.5 * (x1 + x2))

    fraction = float((target - y1) / (y2 - y1))
    fraction = float(np.clip(fraction, 0.0, 1.0))

    if x1 > 0 and x2 > 0:
        return float(np.exp(np.log(x1) + fraction * (np.log(x2) - np.log(x1))))

    return float(x1 + fraction * (x2 - x1))


def _find_profile_crossing(x_values, delta_ts, threshold, side):
    if len(x_values) == 0:
        return None

    minimum_index = int(np.nanargmin(delta_ts))

    if side == 'left':
        for index in range(minimum_index - 1, -1, -1):
            y1 = float(delta_ts[index])
            y2 = float(delta_ts[index + 1])
            if np.isnan(y1) or np.isnan(y2):
                continue
            if np.isclose(y1, threshold):
                return float(x_values[index])
            if (y1 - threshold) * (y2 - threshold) <= 0:
                return _interpolate_profile_crossing(
                    float(x_values[index]),
                    y1,
                    float(x_values[index + 1]),
                    y2,
                    threshold,
                )
        return None

    for index in range(minimum_index, len(x_values) - 1):
        y1 = float(delta_ts[index])
        y2 = float(delta_ts[index + 1])
        if np.isnan(y1) or np.isnan(y2):
            continue
        if np.isclose(y2, threshold):
            return float(x_values[index + 1])
        if (y1 - threshold) * (y2 - threshold) <= 0:
            return _interpolate_profile_crossing(
                float(x_values[index]),
                y1,
                float(x_values[index + 1]),
                y2,
                threshold,
            )

    return None


def _manual_k_profile_interval(
    joint_likelihood,
    parameter_path,
    center_value,
    lower_bound,
    upper_bound,
    n_steps=61,
    target_delta_ts=1.0,
    max_expansions=12,
    scan_expand_factor=1e2,
):
    if center_value <= 0:
        raise ValueError('Manual K profiling requires a positive best-fit normalization.')

    tiny_positive = float(np.finfo(float).tiny)
    scan_min = max(float(lower_bound), tiny_positive)
    scan_max = float(upper_bound) if np.isfinite(upper_bound) else float(center_value) * 1e6

    if not np.isfinite(scan_max) or scan_max <= scan_min:
        raise ValueError('Invalid K scan bounds for manual profiling.')

    import matplotlib.pyplot as plt

    left_crossing = None
    right_crossing = None

    for _ in range(max_expansions + 1):
        x_values, _, profile_values, profile_figure = joint_likelihood.get_contours(
            parameter_path,
            param_1_minimum=float(scan_min),
            param_1_maximum=float(scan_max),
            param_1_n_steps=int(n_steps),
            progress=False,
            log=(True,),
        )
        plt.close(profile_figure)

        x_values = np.asarray(x_values, dtype=float)
        profile_values = np.asarray(profile_values, dtype=float)
        delta_ts = 2.0 * (profile_values - np.nanmin(profile_values))
        minimum_index = int(np.nanargmin(delta_ts))

        left_crossing = _find_profile_crossing(x_values, delta_ts, target_delta_ts, 'left')
        right_crossing = _find_profile_crossing(x_values, delta_ts, target_delta_ts, 'right')

        if left_crossing is not None and right_crossing is not None:
            status = f'manual K profile ok ({len(x_values)} points, {scan_min:.3e}-{scan_max:.3e})'
            return float(left_crossing), float(right_crossing), status

        expanded = False

        if left_crossing is None and minimum_index > 0 and delta_ts[0] < target_delta_ts and scan_min > lower_bound:
            new_scan_min = max(float(lower_bound), float(scan_min) / scan_expand_factor)
            if new_scan_min < scan_min:
                scan_min = new_scan_min
                expanded = True

        if right_crossing is None and minimum_index < (len(x_values) - 1) and delta_ts[-1] < target_delta_ts and scan_max < upper_bound:
            new_scan_max = min(float(upper_bound), float(scan_max) * scan_expand_factor)
            if new_scan_max > scan_max:
                scan_max = new_scan_max
                expanded = True

        if not expanded:
            break

    status = f'manual K profile incomplete ({len(x_values)} points, {scan_min:.3e}-{scan_max:.3e})'
    return left_crossing, right_crossing, status


def fit_thermal_cosi_sed(
    global_results,
    data_hist,
    bkg_hist,
    dr_path,
    sc_orientation,
    source_name=None,
    n_sed_bins=5,
    freeze_cutoff=True,
    freeze_non_target_parameters=True,
    freeze_background=False,
    background_rate_value=None,
    minuit_ftol=None,
    error_estimation='minos_then_manual',
    manual_k_profile_points=61,
    manual_one_sided_mode='upper_limit',
    relax_k_lower_bound=True,
    energy_min_keV=200,
    energy_max_keV=5000,
    scan_expand_factor=1e2,
):
    threeML_config.point_source.integrate_flux_method = 'trapz'

    em_edges = data_hist.axes['Em'].edges.to_value(u.keV)
    native_em_bins = np.flatnonzero(
        (em_edges[:-1] >= energy_min_keV) & (em_edges[1:] <= energy_max_keV)
    )

    if len(native_em_bins) == 0:
        raise ValueError(
            f'No native Em bins are fully contained in {energy_min_keV}-{energy_max_keV} keV.'
        )

    def _split_last_group_remainder(arr, n_groups):
        arr = np.asarray(arr)
        if len(arr) < n_groups:
            raise ValueError(f"Need at least {n_groups} native bins, got {len(arr)}.")

        sizes = [1] * (n_groups - 1)
        sizes.append(len(arr) - (n_groups - 1))

        groups = []
        start = 0
        for size in sizes:
            groups.append(arr[start:start + size])
            start += size

        return groups

    grouped_bins = _split_last_group_remainder(native_em_bins, n_sed_bins)

    sed_rows = []
    best_fit_shape, source_name = _get_cutoff_powerlaw_shape(global_results, source_name=source_name)
    frozen_index = best_fit_shape.index.value
    frozen_cutoff = best_fit_shape.xc.value
    global_k = float(best_fit_shape.K.value)
    global_background_rate = _get_global_background_rate(global_results)

    if background_rate_value is not None:
        background_rate_value = float(background_rate_value)

    for sed_bin_index, group in enumerate(grouped_bins, start=1):
        em_slice = slice(int(group[0]), int(group[-1]) + 1)
        e_min = float(em_edges[group[0]])
        e_max = float(em_edges[group[-1] + 1])
        e_ref = float(np.sqrt(e_min * e_max))
        data_counts = _hist_sum(data_hist.slice[{'Em': em_slice}])
        background_counts = _hist_sum(bkg_hist.slice[{'Em': em_slice}])

        dataset_name = f'cosi_sed_bin_{sed_bin_index}'

        bin_cosi = EnergyRangeCOSIPlugin(
            name=dataset_name,
            dr=dr_path,
            data=data_hist,
            bkg=bkg_hist,
            sc_orientation=sc_orientation,
            em_slice=em_slice,
            nuisance_param=make_cosi_background_parameter(dataset_name),
            earth_occ=True,
        )

        background_parameter = bin_cosi.nuisance_parameters[f'background_{dataset_name}']

        if freeze_background:
            if background_rate_value is None:
                background_parameter.value = global_background_rate
            else:
                background_parameter.value = background_rate_value
            background_parameter.fix = True
        else:
            background_parameter.fix = False
            if background_rate_value is not None:
                background_parameter.value = background_rate_value

        bin_model = build_thermal_sed_model(
            global_results,
            freeze_cutoff=freeze_cutoff,
            source_name=source_name,
            freeze_non_target_parameters=freeze_non_target_parameters,
        )
        bin_like = JointLikelihood(bin_model, DataList(bin_cosi), verbose=False)

        bin_minuit = _make_minuit_with_ftol(minuit_ftol)
        if bin_minuit is not None:
            bin_like.set_minimizer(bin_minuit)

        # threeML can reject every covariance sample in weak bins and then crash while building the summary table.
        fit_status = _run_joint_likelihood_fit(bin_like)

        bin_results = bin_like.results
        fit_shape = _get_source_spectral_shape(bin_results.optimized_model[source_name])
        k_value = float(fit_shape.K.value)
        k_min = float(fit_shape.K.min_value) if fit_shape.K.min_value is not None else 0.0
        k_max = float(fit_shape.K.max_value) if fit_shape.K.max_value is not None else np.inf
        k_lower_bound_relaxed = False

        if relax_k_lower_bound and _hits_lower_bound(k_value, k_min):
            relaxed_k_min = max(k_min * 1e-2, 1e-18)
            if relaxed_k_min < k_min:
                _get_source_spectral_shape(bin_model[source_name]).K.min_value = relaxed_k_min
                retry_status = _run_joint_likelihood_fit(bin_like)
                fit_status = f'{fit_status}; relaxed K min to {relaxed_k_min:.3e}; retry={retry_status}'
                bin_results = bin_like.results
                fit_shape = _get_source_spectral_shape(bin_results.optimized_model[source_name])
                k_value = float(fit_shape.K.value)
                k_min = float(fit_shape.K.min_value) if fit_shape.K.min_value is not None else 0.0
                k_max = float(fit_shape.K.max_value) if fit_shape.K.max_value is not None else np.inf
                k_lower_bound_relaxed = True

        k_parameter_path = fit_shape.K.path
        joint_statistic = _get_joint_likelihood_statistic(bin_like)

        null_dataset_name = f'{dataset_name}_null'
        null_cosi = EnergyRangeCOSIPlugin(
            name=null_dataset_name,
            dr=dr_path,
            data=data_hist,
            bkg=bkg_hist,
            sc_orientation=sc_orientation,
            em_slice=em_slice,
            nuisance_param=make_cosi_background_parameter(null_dataset_name),
            earth_occ=True,
        )

        null_background_parameter = null_cosi.nuisance_parameters[f'background_{null_dataset_name}']
        if freeze_background:
            if background_rate_value is None:
                null_background_parameter.value = global_background_rate
            else:
                null_background_parameter.value = background_rate_value
            null_background_parameter.fix = True
        else:
            null_background_parameter.fix = False
            if background_rate_value is not None:
                null_background_parameter.value = background_rate_value
            else:
                null_background_parameter.value = float(background_parameter.value)

        null_model = _build_null_sed_model(
            global_results,
            source_name=source_name,
            freeze_cutoff=freeze_cutoff,
            freeze_non_target_parameters=freeze_non_target_parameters,
        )
        null_like = JointLikelihood(null_model, DataList(null_cosi), verbose=False)

        null_minuit = _make_minuit_with_ftol(minuit_ftol)
        if null_minuit is not None:
            null_like.set_minimizer(null_minuit)

        null_fit_status = _run_joint_likelihood_fit(null_like)
        joint_null_statistic = _get_joint_likelihood_statistic(null_like)
        ts_value_raw = 2.0 * (joint_null_statistic - joint_statistic)
        ts_value = max(0.0, float(ts_value_raw))
        sigma = _ts_to_sigma(ts_value)

        k_lo = k_value
        k_hi = k_value
        profile_error_status = 'not available'
        error_source = 'profile'
        has_two_sided_error = False

        run_manual_profile = error_estimation == 'manual_profile'

        if error_estimation in ('minos', 'minos_then_manual'):
            try:
                profile_errors = bin_like.get_errors(quiet=True)
                k_errors = _extract_profile_error(profile_errors, k_parameter_path)
                if k_errors is not None:
                    k_lo = max(k_value + float(k_errors[0]), k_min)
                    k_hi = min(k_value + float(k_errors[1]), k_max)
                    profile_error_status = 'ok'
                    error_source = 'profile'
                    has_two_sided_error = bool(np.isfinite(k_lo) and np.isfinite(k_hi) and (k_hi > k_lo))
                    if error_estimation == 'minos_then_manual' and not has_two_sided_error:
                        run_manual_profile = True
                else:
                    profile_error_status = 'K profile error unavailable'
                    if error_estimation == 'minos_then_manual':
                        run_manual_profile = True
            except Exception as exc:
                profile_error_status = f'{type(exc).__name__}: {exc}'
                error_source = 'profile failed'

                if not _hits_lower_bound(k_value, k_min):
                    try:
                        covariance_errors = bin_results.get_data_frame(error_type='covariance')
                        k_cov_errors = _extract_profile_error(covariance_errors, k_parameter_path)
                        if k_cov_errors is not None:
                            cov_lo = max(k_value + float(k_cov_errors[0]), k_min)
                            cov_hi = min(k_value + float(k_cov_errors[1]), k_max)
                            if np.isfinite(cov_lo) and np.isfinite(cov_hi) and (cov_hi > cov_lo):
                                k_lo = cov_lo
                                k_hi = cov_hi
                                profile_error_status = f'covariance fallback ({type(exc).__name__})'
                                error_source = 'covariance fallback'
                                has_two_sided_error = True
                    except Exception:
                        pass

                if error_estimation == 'minos_then_manual' and not has_two_sided_error:
                    run_manual_profile = True

        if run_manual_profile:
            profile_k_max = max(k_max, global_k * 1e6, k_value * 1e6)
            if np.isfinite(profile_k_max) and profile_k_max > k_max:
                profile_shape = _get_source_spectral_shape(bin_model[source_name])
                profile_shape.K.max_value = profile_k_max
                k_max = float(profile_shape.K.max_value)

            k_path = k_parameter_path
            try:
                profile_k_lo, profile_k_hi, manual_status = _manual_k_profile_interval(
                    bin_like,
                    k_path,
                    k_value,
                    k_min,
                    k_max,
                    n_steps=manual_k_profile_points,
                    max_expansions=12,
                    scan_expand_factor=scan_expand_factor,
                )

                if profile_k_lo is not None and profile_k_hi is not None and profile_k_hi > profile_k_lo:
                    k_lo = max(float(profile_k_lo), k_min)
                    k_hi = min(float(profile_k_hi), k_max)
                    profile_error_status = manual_status
                    error_source = 'manual profile'
                    has_two_sided_error = True
                elif profile_k_hi is not None:
                    k_hi = min(float(profile_k_hi), k_max)
                    if manual_one_sided_mode == 'lower_bound_interval':
                        k_lo = k_min
                        profile_error_status = manual_status.replace('incomplete', 'lower-bound interval')
                        error_source = 'manual lower-bound interval'
                        has_two_sided_error = True
                    else:
                        k_lo = k_value
                        profile_error_status = manual_status.replace('incomplete', 'upper limit')
                        error_source = 'manual upper limit'
                else:
                    profile_error_status = manual_status
                    error_source = 'manual profile incomplete'
            except Exception as exc:
                profile_error_status = f'manual K profile failed ({type(exc).__name__}: {exc})'
                error_source = 'manual profile failed'

        integrated_flux = _integrate_model_energy_flux(bin_results.optimized_model, source_name, k_value, e_min, e_max)
        integrated_flux_lo = _integrate_model_energy_flux(bin_results.optimized_model, source_name, k_lo, e_min, e_max)
        integrated_flux_hi = _integrate_model_energy_flux(bin_results.optimized_model, source_name, k_hi, e_min, e_max)
        sed_flux = _evaluate_model_sed_flux(bin_results.optimized_model, source_name, k_value, e_ref)
        sed_flux_lo = _evaluate_model_sed_flux(bin_results.optimized_model, source_name, k_lo, e_ref)
        sed_flux_hi = _evaluate_model_sed_flux(bin_results.optimized_model, source_name, k_hi, e_ref)
        background_rate = float(bin_cosi.nuisance_parameters[f'background_{dataset_name}'].value)
        background_is_fixed = bool(background_parameter.fix)
        hit_parameter_bound = bool(np.isclose(k_value, k_min) or (np.isfinite(k_max) and np.isclose(k_value, k_max)))
        is_upper_limit = bool(
            error_source == 'manual upper limit'
            or ((not has_two_sided_error) and _hits_lower_bound(k_value, k_min))
        )

        sed_rows.append(
            {
                'bin_index': sed_bin_index,
                'source_name': source_name,
                'e_min_keV': e_min,
                'e_max_keV': e_max,
                'e_ref_keV': e_ref,
                'bin_energy_flux_erg_cm2_s': integrated_flux,
                'bin_energy_flux_lo_erg_cm2_s': integrated_flux_lo,
                'bin_energy_flux_hi_erg_cm2_s': integrated_flux_hi,
                'sed_erg_cm2_s': sed_flux,
                'sed_lo_erg_cm2_s': sed_flux_lo,
                'sed_hi_erg_cm2_s': sed_flux_hi,
                'source_K': k_value,
                'source_K_lo': k_lo,
                'source_K_hi': k_hi,
                'source_K_over_global': k_value / global_k if global_k > 0 else np.nan,
                'background_rate_hz': background_rate,
                'background_rate_reference_hz': (
                    global_background_rate if background_rate_value is None else background_rate_value
                ),
                'background_requested_fixed': freeze_background,
                'background_rate_fixed': background_is_fixed,
                'background_released_for_retry': False,
                'data_counts': data_counts,
                'background_counts': background_counts,
                'excess_counts': data_counts - background_counts,
                'joint_statistic': joint_statistic,
                'joint_null_statistic': joint_null_statistic,
                'ts_value_raw': ts_value_raw,
                'ts_value': ts_value,
                'sigma': sigma,
                'null_fit_status': null_fit_status,
                'fit_status': fit_status,
                'profile_error_status': profile_error_status,
                'error_source': error_source,
                'minuit_ftol': minuit_ftol,
                'error_estimation': error_estimation,
                'manual_one_sided_mode': manual_one_sided_mode,
                'k_lower_bound_relaxed': k_lower_bound_relaxed,
                'hit_parameter_bound': hit_parameter_bound,
                'is_upper_limit': is_upper_limit,
                'frozen_index': frozen_index,
                'frozen_cutoff_keV': frozen_cutoff,
            }
        )

    sed_df = pd.DataFrame(sed_rows)
    energy_flux_columns = [
        column for column in sed_df.columns
        if column.endswith("_erg_cm2_s")
    ]
    sed_df[energy_flux_columns] *= SED_KEV_TO_ERG
    return sed_df

In [ ]:
n_thermal_sed_bins = 4

thermal_cosi_sed_df = fit_thermal_cosi_sed(
    global_results=results_sed,
    data_hist=data_sed_hist,
    bkg_hist=bkg_sed_hist,
    dr_path=dr,
    sc_orientation=sc_orientation,
    source_name="NGC1068",
    n_sed_bins=n_thermal_sed_bins,
    freeze_cutoff=True,
    freeze_background=True,
    minuit_ftol=1e-2,
    error_estimation="minos_then_manual",
    manual_k_profile_points=201,
    manual_one_sided_mode="lower_bound_interval",
    energy_min_keV=200,
    energy_max_keV=5000,
    scan_expand_factor=1e6,
)

display(
    thermal_cosi_sed_df[
        [
            "bin_index",
            "e_min_keV",
            "e_max_keV",
            "e_ref_keV",
            "bin_energy_flux_erg_cm2_s",
            "bin_energy_flux_lo_erg_cm2_s",
            "bin_energy_flux_hi_erg_cm2_s",
        ]
    ]
)

display(
    thermal_cosi_sed_df[
        [
            "bin_index",
            "source_K",
            "source_K_lo",
            "source_K_hi",
            "source_K_over_global",
            "background_rate_hz",
            "background_rate_reference_hz",
            "data_counts",
            "background_counts",
            "excess_counts",
            "ts_value",
            "sigma",
            "fit_status",
            "profile_error_status",
            "error_source",
            "hit_parameter_bound",
            "is_upper_limit",
        ]
    ]
)


In [ ]:
FONT_SIZE = 25


def _make_xerr(dataframe):
    return KEV_TO_MEV * np.vstack([
        dataframe["e_ref_keV"] - dataframe["e_min_keV"],
        dataframe["e_max_keV"] - dataframe["e_ref_keV"],
    ])


def _split_sed_detections(dataframe, ts_threshold=4.0):
    upper_limit_mask = dataframe["is_upper_limit"].copy()
    if "ts_value" in dataframe.columns:
        upper_limit_mask |= dataframe["ts_value"] < ts_threshold
    return dataframe.loc[~upper_limit_mask].copy(), dataframe.loc[upper_limit_mask].copy()


fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
ax.tick_params(which="major", size=12, width=1.5, direction="in", top=True, right=True)
ax.tick_params(which="minor", size=6, width=1.5, direction="in", top=True, right=True)
ax.spines["right"].set_visible(True)
ax.spines["top"].set_visible(True)

detections, upper_limits = _split_sed_detections(thermal_cosi_sed_df)
color = "#D55E00"

ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy**2 * flux_inj,
    color=color,
    ls=":",
    lw=3,
    label=r"Injected CPL + PL ($E_c=128$ keV)",
)
ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy**2 * flux_median,
    color=color,
    lw=2,
    label="Global COSI-only fit",
)
ax.fill_between(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy**2 * flux_lo,
    SED_KEV_TO_ERG * energy**2 * flux_hi,
    alpha=0.18,
    color=color,
)

if not detections.empty:
    detection_yerr = np.vstack([
        np.maximum(detections["sed_erg_cm2_s"] - detections["sed_lo_erg_cm2_s"], 0),
        np.maximum(detections["sed_hi_erg_cm2_s"] - detections["sed_erg_cm2_s"], 0),
    ])
    ax.errorbar(
        detections["e_ref_keV"] * KEV_TO_MEV,
        detections["sed_erg_cm2_s"],
        xerr=_make_xerr(detections),
        yerr=detection_yerr,
        fmt="o",
        color=color,
        markerfacecolor="white",
        capsize=4,
        label="COSI SED",
    )

if not upper_limits.empty:
    upper_y = upper_limits["sed_hi_erg_cm2_s"].to_numpy()
    ax.errorbar(
        upper_limits["e_ref_keV"] * KEV_TO_MEV,
        upper_y,
        xerr=_make_xerr(upper_limits),
        yerr=np.maximum(0.5 * upper_y, np.finfo(float).tiny),
        uplims=True,
        fmt="v",
        color=color,
        markerfacecolor="white",
        capsize=6,
        label="_nolegend_",
    )

ax.text(
    0.98,
    0.98,
    f"NGC 1068 (CPL + PL)\n{exposure_1068}-month",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=FONT_SIZE,
    fontweight="550",
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim((85) * KEV_TO_MEV, (12000) * KEV_TO_MEV)
ax.set_ylim(1e-4 * SED_KEV_TO_ERG, 1 * SED_KEV_TO_ERG)
ax.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE)
ax.set_ylabel(r"Energy Flux (erg cm$^{-2}$ s$^{-1}$)", fontsize=FONT_SIZE)
ax.legend(fontsize=FONT_SIZE, loc="lower left", frameon=False)

save_path = (
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    f"Papers/AGN_Corona_EC/Plots/NGC1068_CPL_PL_128_{exposure_1068}Months_SED.pdf"
)
# plt.savefig(save_path)
